# Project 2 — Feature Engineering, ML, Risk Scoring & Audit

This notebook turns cleaned transaction data into behavioral features, trains an Isolation Forest anomaly detector, applies explainable business rules, combines them into a configurable risk score, creates an audit queue, and evaluates the model using **separate synthetic evaluation labels only after scoring**.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT=Path.cwd()
while PROJECT_ROOT!=PROJECT_ROOT.parent and not (PROJECT_ROOT/"data"/"raw").exists():
    PROJECT_ROOT=PROJECT_ROOT.parent
sys.path.insert(0,str(PROJECT_ROOT))

from src.config import RAW_DIR, PROCESSED_DIR, REFERENCE_DIR, REPORT_DIR, MODEL_DIR
from src.config import MODEL_CONTAMINATION, RANDOM_STATE, TRAIN_END, HIGH_RISK_THRESHOLD, MEDIUM_RISK_THRESHOLD
from src.io_utils import read_raw_tables
from src.cleaning.clean import clean_tables
from src.features.build_features import build_transaction_features
from src.model.train import train_and_score, evaluate
from src.risk.rules import apply_business_rules
from src.risk.scoring import build_risk_scores
from src.risk.audit_cases import build_audit_cases

tables=read_raw_tables(RAW_DIR)
cleaned,_=clean_tables(tables,PROCESSED_DIR)
features=build_transaction_features(cleaned["customers"],cleaned["accounts"],cleaned["merchants"],cleaned["customer_devices"],cleaned["transactions"])
print("Feature rows:",len(features))

## 1. Behavioral features

In [ ]:
feature_cols=[
    "transaction_id","amount","customer_median_amount","amount_robust_z",
    "amount_to_customer_median","tx_count_10m","tx_count_1h","tx_count_24h",
    "minutes_since_prev_tx","location_distance_km","location_change",
    "new_device","duplicate_candidate_2min","channel_prior_share",
    "merchant_category_prior_share"
]
display(features[feature_cols].head(20))

## 2. Isolation Forest

The ground-truth evaluation file is not supplied to model training.

In [ ]:
ml_scores=train_and_score(features,TRAIN_END,MODEL_DIR,MODEL_CONTAMINATION,RANDOM_STATE)
scored=features.merge(ml_scores,on="transaction_id")
display(scored[["transaction_id","ml_anomaly_score"]].sort_values("ml_anomaly_score",ascending=False).head(25))

## 3. Explainable business rules

In [ ]:
rules=apply_business_rules(scored)
display(rules[["transaction_id","rule_count","rule_score","rule_reason"]].sort_values(["rule_count","rule_score"],ascending=False).head(25))

## 4. Combined risk score

In [ ]:
risk=build_risk_scores(rules,HIGH_RISK_THRESHOLD,MEDIUM_RISK_THRESHOLD)
print(risk["risk_level"].value_counts())
display(risk[["transaction_id","ml_anomaly_score","rule_score","behavior_score","context_score","risk_score","risk_level","risk_reason"]].sort_values("risk_score",ascending=False).head(25))

## 5. Audit queue

In [ ]:
cases=build_audit_cases(risk)
print("Audit cases:",len(cases))
display(cases.head(25))

## 6. Model evaluation using the separate synthetic ground truth

These labels are used **only after scoring** for development evaluation. They do not become model features.

In [ ]:
truth=pd.read_csv(REFERENCE_DIR/"transaction_ground_truth.csv")
metrics,scenario=evaluate(ml_scores,truth)
print("Model evaluation")
display(metrics)
print("Scenario breakdown")
display(scenario)

## 7. Save modeling outputs

In [ ]:
features.to_csv(PROCESSED_DIR/"transaction_features.csv",index=False)
risk.to_csv(PROCESSED_DIR/"risk_predictions.csv",index=False)
cases.to_csv(PROCESSED_DIR/"audit_cases.csv",index=False)
print("Outputs saved")

## Final note

An anomaly score means **unusual relative to learned transaction behavior**. It does not prove fraud, wrongdoing, or customer intent. A real deployment would require validation, governance, monitoring, human review and threshold calibration.